# T06: Star Schema Export

## What You'll Learn

Spindle generates **normalized 3NF data** by default, but many analytics tools — Power BI,
Fabric DirectLake, traditional data warehouses — work best with a **star schema**
(dimension + fact tables). In this tutorial you will:

1. **Generate** a retail dataset in 3NF
2. **Transform** it to a star schema using `StarSchemaTransform`
3. **Explore** the dimension tables (`dim_customer`, `dim_product`, `dim_store`)
4. **Explore** the fact tables (`fact_sale`, `fact_return`)
5. **Inspect** the auto-generated `dim_date` table
6. **Export** everything to Parquet files

## Prerequisites

- `sqllocks-spindle` installed
- Familiarity with `Spindle.generate()` (see **T01**)

## Time Estimate

**~10 minutes**

In [ ]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

## Step 1 — Generate Retail Data and Transform to Star Schema

**What we're about to do:** Generate a retail dataset, then use `StarSchemaTransform`
and the domain's `star_schema_map()` to convert 3NF tables into a star schema.

**Why this matters:** The `star_schema_map()` method on each domain defines how 3NF
tables map to dimensions and facts — which columns become surrogate keys, which
become measures, and how date columns link to `dim_date`. One call handles all of it.

**What to expect:** A summary showing all dimension and fact tables with their row counts.

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain
from sqllocks_spindle.transform import StarSchemaTransform

# Generate 3NF retail data
spindle = Spindle()
result = spindle.generate(domain=RetailDomain(), scale="fabric_demo", seed=42)

# Transform to star schema
transform = StarSchemaTransform()
star = transform.transform(result.tables, RetailDomain().star_schema_map())

print(star.summary())

## Step 2 — Explore Dimension Tables

**What we're about to do:** Inspect `dim_customer`, `dim_product`, and `dim_store`.
Each dimension has a surrogate key (`sk_*`) as its first column and a natural key
(`nk_*`) that maps back to the original 3NF primary key.

**Why this matters:** Surrogate keys are essential for star schema integrity — they
decouple the warehouse from source system IDs. The preserved natural keys let you
trace any row back to its origin if needed.

In [ ]:
# Dimension tables available
print("Dimension Tables:")
for name, df in star.dimensions.items():
    sk_cols = [c for c in df.columns if c.startswith("sk_")]
    nk_cols = [c for c in df.columns if c.startswith("nk_")]
    print(f"  {name:<20} {len(df):>6} rows  |  SK: {sk_cols}  NK: {nk_cols}")

print("\n=== dim_customer (first 5 rows) ===")
print(star.dimensions["dim_customer"].head().to_string(index=False))

print("\n=== dim_product (first 5 rows) ===")
print(star.dimensions["dim_product"].head().to_string(index=False))

print("\n=== dim_store (first 5 rows) ===")
print(star.dimensions["dim_store"].head().to_string(index=False))

## Step 3 — Explore Fact Tables

**What we're about to do:** Look at `fact_sale` and `fact_return`. Fact tables contain
surrogate key columns (`sk_*`) for joining to dimensions, natural keys (`nk_*`) for
traceability, and measure columns (quantities, amounts).

**Why this matters:** In a properly modeled star schema, all analytical queries go
through fact tables. The `sk_date` column uses YYYYMMDD integer format, which lets
you do efficient range filtering and joins directly to `dim_date`.

In [ ]:
# Fact tables available
print("Fact Tables:")
for name, df in star.facts.items():
    sk_cols = [c for c in df.columns if c.startswith("sk_")]
    print(f"  {name:<20} {len(df):>6} rows  |  SK joins: {sk_cols}")

print("\n=== fact_sale (first 5 rows) ===")
print(star.facts["fact_sale"].head().to_string(index=False))

if "fact_return" in star.facts:
    print("\n=== fact_return (first 5 rows) ===")
    print(star.facts["fact_return"].head().to_string(index=False))

## Step 4 — The Auto-Generated dim_date

**What we're about to do:** Inspect the `dim_date` table that `StarSchemaTransform`
automatically generates from the date range found in your fact data.

**Why this matters:** Every date dimension needs calendar attributes — year, quarter,
month name, day of week, fiscal periods, is_weekend flags. Spindle builds all of this
automatically so you don't have to maintain a separate date utility.

In [ ]:
dim_date = star.date_dim

print(f"Date range: {dim_date['date'].min()} to {dim_date['date'].max()}")
print(f"Total days: {len(dim_date):,}")
print(f"\nColumns ({len(dim_date.columns)}):")
print(f"  {list(dim_date.columns)}")

print("\n=== dim_date (first 10 rows) ===")
print(dim_date.head(10).to_string(index=False))

## Step 5 — Export to Parquet Files

**What we're about to do:** Write every dimension, fact, and the date dimension to
individual Parquet files. Parquet is the standard columnar format for Lakehouse,
Spark, and Power BI DirectLake workloads.

**Why this matters:** Parquet preserves data types, compresses well, and is natively
supported by virtually every analytics engine. This is the fastest path from Spindle
to a production-like Lakehouse.

In [ ]:
import os

output_dir = "./spindle_star_output"
os.makedirs(output_dir, exist_ok=True)

# Write all star schema tables (dimensions, facts, date dim) to Parquet
for table_name, df in star.all_tables().items():
    path = os.path.join(output_dir, f"{table_name}.parquet")
    df.to_parquet(path, index=False)

files = sorted(os.listdir(output_dir))
print(f"Exported {len(files)} Parquet files to {output_dir}/\n")
for f in files:
    size = os.path.getsize(os.path.join(output_dir, f))
    print(f"  {f:<35} {size:>8,} bytes")

## What's Next?

You've transformed 3NF data into a full star schema and exported it to Parquet. Here's
where to go from here:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T08: Fabric Quickstart** | Write the star schema directly to a Fabric Lakehouse as Delta tables |
| **04: Star Schema (reference)** | Advanced star schema topics including CDM export and healthcare star schema |
| **T05: Distribution Overrides** | Customize the source data distributions before transforming |

**Key takeaways:**
- `StarSchemaTransform` + `domain.star_schema_map()` handles the full 3NF-to-star conversion
- Surrogate keys (`sk_*`) are auto-generated; natural keys (`nk_*`) are preserved
- `dim_date` is automatically built from your fact data's date range
- `star.all_tables()` returns every table (dims + facts + date) in one dict